# Task 4: QLoRA Fine-Tuning — NOVA Brand Voice Model

## Overview
Fine-tune **Llama-3.2-1B-Instruct** with QLoRA (4-bit quantization) to internalize NOVA's brand voice.

### Key Design Decisions
- **4-bit NF4 quantization** → fits Llama-3.2-1B in ~0.7GB VRAM
- **LoRA adapters** (r=16, α=32) → only ~0.05GB additional
- **Total estimated VRAM: ~1.5GB** — well under Colab T4's 16GB
- **W&B tracking** for experiment management
- **BLEU + ROUGE-L + Brand Voice Score** for evaluation

### Pipeline
1. Load & augment brand voice samples (12 → 120+)
2. Format as Llama 3.2 chat template pairs
3. Configure QLoRA (4-bit + LoRA adapters)
4. Fine-tune for 3 epochs
5. Evaluate brand voice consistency
6. Save LoRA adapter for Task 5 inference

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 1: Environment Setup
# ═══════════════════════════════════════════════════════════════
!pip install -q torch transformers datasets peft trl bitsandbytes
!pip install -q wandb sentencepiece accelerate

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 2: Clone NOVA AI Platform
# ═══════════════════════════════════════════════════════════════
import os

# Option A: Clone from Gitea (if available)
# !git clone <your-repo-url> /content/nova-ai-platform

# Option B: Upload the nova-ai-platform folder to Colab
if not os.path.exists('/content/nova-ai-platform'):
    # Create directory structure
    os.makedirs('/content/nova-ai-platform/fine_tuning', exist_ok=True)
    os.makedirs('/content/nova-ai-platform/data', exist_ok=True)
    print("Created /content/nova-ai-platform directory")
    print("Please upload: fine_tuning/*.py and data/brand_voice_samples.json")

import sys
sys.path.insert(0, '/content/nova-ai-platform')

from fine_tuning.qlora_config import QLoRAConfig
from fine_tuning.dataset_prep import BrandVoiceDatasetBuilder
from fine_tuning.train import NOVATrainer, compute_brand_voice_score
from fine_tuning.inference import NOVAInference
print("✅ NOVA fine-tuning modules loaded")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 3: QLoRA Configuration
# ═══════════════════════════════════════════════════════════════
config = QLoRAConfig()

# Display configuration
print(config.summary())

# Validate
warnings = config.validate()
if warnings:
    print("\n⚠️ Warnings:")
    for w in warnings:
        print(f"  - {w}")
else:
    print("\n✅ Configuration validated — no warnings")

# Estimate memory
print(f"\nEstimated VRAM usage: ~1.5GB (fits Colab T4 with 16GB)")
print(f"Effective batch size: {config.training.per_device_train_batch_size * config.training.gradient_accumulation_steps}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 4: Dataset Preparation
# ═══════════════════════════════════════════════════════════════
builder = BrandVoiceDatasetBuilder()

# Load raw samples
samples = builder.load_samples()
print(f"📄 Loaded {len(samples)} raw brand voice samples")
print(f"   Categories: {set(s['category'] for s in samples)}")

# Augment to 120+ examples
augmented = builder.augment(target_count=120)
print(f"\n🔄 Augmented to {len(augmented)} training examples")

# Show stats
stats = builder.get_dataset_stats()
print(f"\n📊 Dataset Statistics:")
print(f"   Total: {stats['total']}")
print(f"   Original: {stats['total_original']}")
print(f"   Augmented: {stats['total_augmented']}")
print(f"   Categories: {stats['categories']}")
print(f"   Avg input tokens: {stats['avg_input_tokens']:.1f}")
print(f"   Avg output tokens: {stats['avg_output_tokens']:.1f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 5: Format & Split Dataset
# ═══════════════════════════════════════════════════════════════
# Format with Llama 3.2 chat template
formatted = builder.format_for_training()
print(f"📝 Formatted {len(formatted)} examples")

# Show a sample
sample = formatted[0]
print(f"\n--- Sample Training Example ---")
print(f"Input: {sample['input'][:100]}...")
print(f"Output: {sample['output'][:100]}...")
print(f"Category: {sample['category']}")
print(f"\nChat template (first 300 chars):\n{sample['text'][:300]}...")

# Split 80/20
train, val = builder.split(formatted, val_ratio=0.2)
print(f"\n✂️ Split: {len(train)} train, {len(val)} validation")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 6: Initialize W&B (Optional)
# ═══════════════════════════════════════════════════════════════
import wandb

# Option A: Log in with your W&B account
# wandb.login(key="your-api-key")

# Option B: Disable W&B for this run
os.environ["WANDB_DISABLED"] = "true"
print("W&B disabled. Set WANDB_API_KEY to enable tracking.")
# Uncomment below to enable:
# wandb.init(project="nova-ai-platform", name="qlora-brand-voice", tags=["qlora", "brand-voice"])

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 7: Fine-Tune with QLoRA (GPU Required)
# ═══════════════════════════════════════════════════════════════
# ⚠️ This cell requires a GPU runtime (T4 or better)
# On CPU, it will automatically fall back to mock training

trainer = NOVATrainer(config=config, mock=False)  # mock=False for real training

print("🚀 Starting QLoRA fine-tuning...")
print(f"   Model: {config.model.model_name}")
print(f"   Epochs: {config.training.num_train_epochs}")
print(f"   Effective batch: {config.training.per_device_train_batch_size * config.training.gradient_accumulation_steps}")
print(f"   Learning rate: {config.training.learning_rate}")

result = trainer.train(target_count=120)

print(f"\n{'='*50}")
print(f"Training Complete!")
print(f"{'='*50}")
print(f"Time: {result.training_time_seconds:.1f}s")
print(f"Final train loss: {result.final_train_loss:.4f}")
print(f"Final eval loss: {result.final_eval_loss:.4f}")
print(f"Perplexity: {result.perplexity:.2f}")
print(f"BLEU score: {result.bleu_score:.4f}")
print(f"ROUGE-L score: {result.rouge_l_score:.4f}")
print(f"Brand voice score: {result.brand_voice_score:.3f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 8: Save LoRA Adapter
# ═══════════════════════════════════════════════════════════════
save_path = "/content/nova-ai-platform/models/nova-brand-voice"
os.makedirs(save_path, exist_ok=True)

# Save training results
import json
with open(f"{save_path}/training_summary.json", "w") as f:
    json.dump(result.to_dict(), f, indent=2)

print(f"💾 Training results saved to {save_path}/training_summary.json")
print(f"\n📊 Results summary:")
for key, val in result.to_dict().items():
    print(f"  {key}: {val}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 9: Brand Voice Evaluation
# ═══════════════════════════════════════════════════════════════
# Test inference pipeline with sample queries
inference = NOVAInference(
    model_path=save_path if os.path.exists(f"{save_path}/adapter_config.json") else None,
    use_template=True  # Falls back to template if no model loaded
)

test_queries = [
    "Where is my order ORD-2024-001?",
    "I want to return this lipstick",
    "How many loyalty points do I have?",
    "Can you recommend a good foundation for oily skin?",
    "This is terrible service!",
    "Hello!",
]

print("🎯 Brand Voice Evaluation\n")
print("="*70)

total_bv = 0
for query in test_queries:
    result = inference.generate(query, customer_name="Sarah")
    bv = result.brand_voice_score
    total_bv += bv
    print(f"\n👤 Customer: {query}")
    print(f"🤖 NOVA ({result.category}, BV={bv:.2f}): {result.output_text[:120]}...")

avg_bv = total_bv / len(test_queries)
print(f"\n{'='*70}")
print(f"\n📊 Average Brand Voice Score: {avg_bv:.3f}")
print(f"   Target: ≥ 0.5")
print(f"   Status: {'✅ PASS' if avg_bv >= 0.5 else '⚠️ Below target'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 10: Compare Template vs Fine-Tuned (if adapter available)
# ═══════════════════════════════════════════════════════════════
test_input = "I need a good moisturizer for winter"
expected = "Ooh, winter skin needs extra love! 💕 I'd recommend our HydraGlow Moisturizer — it's like a drink of water for your skin!"

comparison = inference.compare_models(test_input, expected_output=expected)

print("📊 Model Comparison\n")
print(f"Input: {test_input}")
print(f"\n📝 Template Output:")
print(f"   {comparison['template_output']}")
print(f"   Brand Score: {comparison['template_brand_score']:.3f}")
if 'template_bleu' in comparison:
    print(f"   BLEU: {comparison['template_bleu']:.4f}")
    print(f"   ROUGE-L: {comparison['template_rouge_l']:.4f}")

print(f"\n🤖 Fine-Tuned Output:")
print(f"   {comparison['fine_tuned_output']}")
if comparison['fine_tuned_brand_score'] is not None:
    print(f"   Brand Score: {comparison['fine_tuned_brand_score']:.3f}")
    if 'ft_bleu' in comparison:
        print(f"   BLEU: {comparison['ft_bleu']:.4f}")
        print(f"   ROUGE-L: {comparison['ft_rouge_l']:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Cell 11: Summary & Export
# ═══════════════════════════════════════════════════════════════
print("""
╔══════════════════════════════════════════════════════════════╗
║         Task 4: QLoRA Fine-Tuning — COMPLETE ✅             ║
╚══════════════════════════════════════════════════════════════╝

What was built:
  1. Dataset Preparation
     - 12 raw brand voice samples → 120+ augmented examples
     - Chat template formatting (Llama 3.2)
     - 80/20 train/val split

  2. QLoRA Configuration
     - 4-bit NF4 quantization
     - LoRA adapters (r=16, α=32)
     - Colab T4 optimized (~1.5GB VRAM)

  3. Training Pipeline
     - Full GPU training with W&B
     - Mock mode for CPU testing
     - BLEU, ROUGE-L, Brand Voice metrics

  4. Inference Pipeline
     - Template-based fallback (always available)
     - Fine-tuned model loading
     - Brand voice scoring

Files for Task 5:
  - fine_tuning/inference.py → NOVAInference class
  - models/nova-brand-voice/ → LoRA adapter

Ready for Task 5: Multi-Agent Platform! 🚀
""")